In [15]:
import numpy as np
import pandas as pd
import random

In [16]:
def calculate_similarity(arr, lamda):
    return (np.sum(arr)**2)/(len(arr) + lamda )

def best_gain(X, y, lamda, gamma):
    best_score, best_col, best_thresh = -np.inf, None, None
    root_similarity = calculate_similarity(y, lamda)

    for col in range(X.shape[1]):
        s = np.sort(np.unique(X[:, col]))
        thresholds = [(s[i] + s[i+1])/2 for i in range(len(s) - 1)]

        for thresh in thresholds:
            score = split_score(X, y, col,root_similarity, thresh, lamda)
            if score > gamma and score > best_score:
                best_score, best_col, best_thresh = score, col, thresh
                
    return (best_col, best_score, best_thresh)
        
def split_score(X, y, feature_column, root_similarity, threshold, lamda):
    true_array = y[X[:, feature_column] >= threshold]
    false_array = y[X[:, feature_column] < threshold]
    true_res = calculate_similarity(true_array, lamda)
    false_res = calculate_similarity(false_array, lamda)

    return true_res + false_res - root_similarity

In [17]:
from __future__ import annotations
from dataclasses import dataclass

@dataclass
class LeafNode:
    res: float

@dataclass
class DecisionNode:
    col: int
    threshold: float
    leftNode: DecisionNode | LeafNode | None = None
    rightNode: DecisionNode | LeafNode | None = None

In [18]:
def _build(X, y, depth = 2, score_cutoff = 0, lamda = 1, gamma = 0):   
    if not len(X) or not len(y):
        raise IndexError("Length was 0?")
    
    best_col, best_score, thresh = best_gain(X, y, lamda, gamma)
    curr = DecisionNode(best_col, thresh)

    if depth == 0 or best_col == None:
        mean = np.sum(y)/(len(y) + lamda)
        return LeafNode(mean)
    else:
        left_x, left_y = X[X[:,best_col] >= thresh], y[X[:,best_col] >= thresh]
        right_x, right_y = X[X[:,best_col] < thresh], y[X[:,best_col] < thresh]
        curr.leftNode = _build(left_x, left_y, depth-1, score_cutoff, lamda=lamda, gamma=gamma)
        curr.rightNode = _build(right_x, right_y, depth-1, score_cutoff, lamda=lamda, gamma=gamma)
    return curr

def predict(X, tree):
    ret = []
    for sample in X:
        curr = tree
        while curr and type(curr) is DecisionNode:
            c = curr.col
            if sample[c] >= curr.threshold:
                curr = curr.leftNode
            else:
                curr = curr.rightNode
        ret.append(curr.res if curr else None)
    return ret

def create_xgboost(X, y, depth, num_trees = 3, lamda = 0, gamma = 0):
    ret = [np.average(y)]
    for _ in range(num_trees):
        new_y = y - xgboost_predict(X,ret)
        ret.append(_build(X, new_y, depth = depth, lamda=lamda, gamma=gamma))
    return ret

def xgboost_predict(X, f, lr = 0.1):
    curr = np.zeros(X.shape[0])
    for tree in f[1:]:
        curr += predict(X, tree)
    return f[0] + lr * curr 

In [19]:
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestClassifier
from collections import deque

def _display(curr: DecisionNode | LeafNode | None):
    if type(curr) is DecisionNode:
        print(f"col: {curr.col}, threshold: {curr.threshold:.2f}", end = " | ")
    elif type(curr) is LeafNode:
        print(f"res: {curr.res}", end = " | ")
    else:
        print(f"res : {None}")
        
def _display_graph(tree):
    q, newq = deque([tree]), deque([])
    while q or newq:
        while q:
            curr = q.pop()
            _display(curr)
            if not curr or type(curr) is not DecisionNode: continue
            for item in [curr.leftNode, curr.rightNode]:
                newq.appendleft(item)
        q, newq = newq, deque()
        print("", end ="\n")

In [20]:
X_rt = np.array([
    [1.0, 5.0],
    [2.0, 3.0],
    [3.0, 4.0],
    [4.0, 1.0],
    [5.0, 2.0],
    [6.0, 6.0],
])
y_rt = np.array([1.0, 2.0, 1.5, 9.0, 8.5, 10.0])

LEARNING_RATE = 0.1
local_gb = create_xgboost(X_rt, y_rt, depth = 3,num_trees = 3, lamda = 0, gamma = 0)
res = xgboost_predict(X_rt, local_gb)
print(res)
_display_graph(local_gb[1])
len(res)

[4.159  4.43   4.2945 6.327  6.1915 6.598 ]
col: 0, threshold: 3.50 | 
col: 0, threshold: 5.50 | col: 0, threshold: 1.50 | 
res: 4.666666666666667 | col: 0, threshold: 4.50 | col: 0, threshold: 2.50 | res: -4.333333333333333 | 
res: 3.166666666666667 | res: 3.666666666666667 | res: -3.833333333333333 | res: -3.333333333333333 | 


6

In [22]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

X, y = make_regression(n_samples=300, n_features=5, noise=10.0, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

xgb = XGBRegressor(
    n_estimators=100, learning_rate=0.1, max_depth=3,
    reg_lambda=1.0, gamma=0.0, min_child_weight=1.0,
    tree_method="exact",
    base_score=float(y_train.mean()),   # a probability; xgboost converts it to log-odds
)

xgb.fit(X_train, y_train)
local_gb = create_xgboost(X_train, y_train, num_trees=100, depth = 3, lamda = 1, gamma= 0)

res = xgboost_predict(X_test, local_gb)
sk_preds = xgb.predict(X_test)
# print(sk_preds)
# print(res)
print(mean_squared_error(y_test, sk_preds), r2_score(y_test, sk_preds))
print(mean_squared_error(y_test, res), r2_score(y_test, res))

409.4533242375855 0.9631789644857989
409.4533971303718 0.9631789579307468
